# Notebook 01 — Data Loading, Exploration and Preprocessing

## NYC Yellow Taxi Trip Data — January 2022

This notebook is the first step of the Big Data / Machine Learning project.  
The goal is to load the raw NYC Yellow Taxi dataset, inspect its structure, identify missing or invalid values, apply basic preprocessing rules and save cleaned datasets for the next stages of the project.

The processed data generated in this notebook will later be used for:

1. Benchmarking database-like operations across different Python Big Data libraries.
2. Building a machine learning pipeline to predict the target variable `fare_amount`.
3. Creating smaller and larger dataset versions for scalability experiments.

The raw dataset used in this notebook is:

`yellow_tripdata_2022-01.parquet`

In [1]:
# Basic imports
from pathlib import Path
import pandas as pd
import numpy as np
import time
import os
import sys

## 1. Project paths

The project follows a simple and reproducible structure:

```text
nyc_taxi_project/
  data/
    raw/
    processed/
  notebooks/
  results/
  src/

In [3]:

# Define project paths

PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
SRC_DIR = PROJECT_ROOT / "src"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = RAW_DIR / "yellow_tripdata_2022-01.parquet"

print("Project root:", PROJECT_ROOT)
print("Raw file:", RAW_FILE)
print("Processed directory:", PROCESSED_DIR)
print("Results directory:", RESULTS_DIR)

Project root: /Users/nunoantunes/Desktop/nyc_taxi_project
Raw file: /Users/nunoantunes/Desktop/nyc_taxi_project/data/raw/yellow_tripdata_2022-01.parquet
Processed directory: /Users/nunoantunes/Desktop/nyc_taxi_project/data/processed
Results directory: /Users/nunoantunes/Desktop/nyc_taxi_project/results


In [4]:
# Check if the raw file exists

if RAW_FILE.exists():
    file_size_mb = RAW_FILE.stat().st_size / (1024 ** 2)
    print(f"File found: {RAW_FILE.name}")
    print(f"File size: {file_size_mb:.2f} MB")
else:
    raise FileNotFoundError(f"Raw file not found: {RAW_FILE}")

File found: yellow_tripdata_2022-01.parquet
File size: 36.37 MB


## 2. Load the raw dataset

The dataset is stored in Parquet format.  
Parquet is a columnar storage format, which is commonly used in Big Data workflows because it is more efficient than CSV for reading selected columns and handling large datasets.

In [5]:
# Load dataset and measure reading time

start_time = time.perf_counter()

df_raw = pd.read_parquet(RAW_FILE)

end_time = time.perf_counter()
read_time = end_time - start_time

print(f"Dataset loaded successfully.")
print(f"Reading time: {read_time:.4f} seconds")
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

Dataset loaded successfully.
Reading time: 0.1737 seconds
Shape: 2,463,931 rows x 19 columns


In [6]:
# Display first rows

df_raw.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.80,1.0,N,142,236,1,14.5,3.0,0.5,3.65,0.0,0.3,21.95,2.5,0.0
1,1,2022-01-01 00:33:43,2022-01-01 00:42:07,1.0,2.10,1.0,N,236,42,1,8.0,0.5,0.5,4.00,0.0,0.3,13.30,0.0,0.0
2,2,2022-01-01 00:53:21,2022-01-01 01:02:19,1.0,0.97,1.0,N,166,166,1,7.5,0.5,0.5,1.76,0.0,0.3,10.56,0.0,0.0
3,2,2022-01-01 00:25:21,2022-01-01 00:35:23,1.0,1.09,1.0,N,114,68,2,8.0,0.5,0.5,0.00,0.0,0.3,11.80,2.5,0.0
4,2,2022-01-01 00:36:48,2022-01-01 01:14:20,1.0,4.30,1.0,N,68,163,1,23.5,0.5,0.5,3.00,0.0,0.3,30.30,2.5,0.0


## 3. Dataset structure

In this section, the structure of the dataset is analyzed.  
This includes the number of rows and columns, column names, data types and memory usage.

In [7]:
# Dataset shape

num_rows, num_cols = df_raw.shape

print(f"Number of rows: {num_rows:,}")
print(f"Number of columns: {num_cols}")

Number of rows: 2,463,931
Number of columns: 19


In [8]:
# Column names

df_raw.columns.tolist()

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'airport_fee']

In [9]:
# Data types

df_raw.dtypes

VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                     float64
dtype: object

In [10]:
# General information about the DataFrame

df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2463931 entries, 0 to 2463930
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int64         
 8   DOLocationID           int64         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  airport_fee           

In [11]:
# Memory usage

memory_mb = df_raw.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"Approximate memory usage: {memory_mb:.2f} MB")

Approximate memory usage: 472.34 MB


## 4. Descriptive statistics

The next step is to analyze the numerical variables.  
This helps identify possible outliers, impossible values and variables that may require cleaning.

In [13]:
# Descriptive statistics for numerical columns

df_raw.describe().T

,count,mean,min,25%,50%,75%,max,std
VendorID,2463931.0,1.707819,1.0,1.0,2.0,2.0,6.0,0.502137
tpep_pickup_datetime,2463931,2022-01-17 01:19:51.689724,2008-12-31 22:23:09,2022-01-09 15:37:41,2022-01-17 12:11:45,2022-01-24 13:49:37.500000,2022-05-18 20:41:57,NaN
tpep_dropoff_datetime,2463931,2022-01-17 01:34:04.421902,2008-12-31 23:06:56,2022-01-09 15:50:50.500000,2022-01-17 12:23:49,2022-01-24 14:02:51,2022-05-18 20:47:45,NaN
passenger_count,2392428.0,1.389453,0.0,1.0,1.0,1.0,9.0,0.982969
trip_distance,2463931.0,5.372751,0.0,1.04,1.74,3.13,306159.28,547.871404
RatecodeID,2392428.0,1.415507,1.0,1.0,1.0,1.0,99.0,5.917573
PULocationID,2463931.0,166.076809,1.0,132.0,162.0,234.0,265.0,65.468057
DOLocationID,2463931.0,163.580716,1.0,113.0,162.0,236.0,265.0,70.790159
payment_type,2463931.0,1.194449,0.0,1.0,1.0,1.0,5.0,0.500178
fare_amount,2463931.0,12.946484,-480.0,6.5,9.0,14.0,401092.32,255.814887


## 5. Missing values

Missing values can affect both benchmark operations and machine learning models.  
Here, the number and percentage of missing values per column are computed.

In [15]:
# Missing values summary

missing_summary = pd.DataFrame({
    "missing_values": df_raw.isna().sum(),
    "missing_percentage": (df_raw.isna().mean() * 100).round(2)
})

missing_summary = missing_summary.sort_values(by="missing_values", ascending=False)
missing_summary

,missing_values,missing_percentage
airport_fee,71503,2.9
congestion_surcharge,71503,2.9
passenger_count,71503,2.9
RatecodeID,71503,2.9
store_and_fwd_flag,71503,2.9
extra,0,0.0
total_amount,0,0.0
improvement_surcharge,0,0.0
tolls_amount,0,0.0
tip_amount,0,0.0


## 6. Target variable analysis: `fare_amount`

The target variable for the machine learning task is `fare_amount`.

For regression, the model will predict the numerical fare amount directly.  
For classification, the fare amount can be discretized into categories.

In [16]:
# Basic statistics for the target variable

target = "fare_amount"

if target in df_raw.columns:
    print(df_raw[target].describe())
else:
    raise ValueError(f"Target column '{target}' not found in dataset.")

count    2.463931e+06
mean     1.294648e+01
std      2.558149e+02
min     -4.800000e+02
25%      6.500000e+00
50%      9.000000e+00
75%      1.400000e+01
max      4.010923e+05
Name: fare_amount, dtype: float64


In [17]:
# Count invalid fare_amount values

invalid_fares = df_raw[
    (df_raw["fare_amount"].isna()) |
    (df_raw["fare_amount"] <= 0)
]

print(f"Invalid fare_amount rows: {len(invalid_fares):,}")
print(f"Percentage: {len(invalid_fares) / len(df_raw) * 100:.2f}%")

Invalid fare_amount rows: 13,892
Percentage: 0.56%


In [18]:
# Check fare amount distribution using quantiles

fare_quantiles = df_raw["fare_amount"].quantile([
    0.00, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.00
])

fare_quantiles

0.00      -480.00
0.01         2.50
0.05         4.50
0.25         6.50
0.50         9.00
0.75        14.00
0.95        39.50
0.99        52.50
1.00    401092.32
Name: fare_amount, dtype: float64

## 7. Trip distance analysis

The variable `trip_distance` is one of the most relevant features for predicting `fare_amount`.  
Trips with zero, negative or extremely high distances are likely invalid or abnormal and should be filtered.

In [19]:
# Basic statistics for trip distance

if "trip_distance" in df_raw.columns:
    print(df_raw["trip_distance"].describe())
else:
    print("Column 'trip_distance' not found.")

count    2.463931e+06
mean     5.372751e+00
std      5.478714e+02
min      0.000000e+00
25%      1.040000e+00
50%      1.740000e+00
75%      3.130000e+00
max      3.061593e+05
Name: trip_distance, dtype: float64


In [20]:
# Count invalid trip_distance values

invalid_distances = df_raw[
    (df_raw["trip_distance"].isna()) |
    (df_raw["trip_distance"] <= 0)
]

print(f"Invalid trip_distance rows: {len(invalid_distances):,}")
print(f"Percentage: {len(invalid_distances) / len(df_raw) * 100:.2f}%")

Invalid trip_distance rows: 29,373
Percentage: 1.19%


In [21]:
# Trip distance quantiles

distance_quantiles = df_raw["trip_distance"].quantile([
    0.00, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.00
])

distance_quantiles

0.00         0.00
0.01         0.00
0.05         0.50
0.25         1.04
0.50         1.74
0.75         3.13
0.95        11.80
0.99        19.70
1.00    306159.28
Name: trip_distance, dtype: float64

## 8. Date and time variables

The dataset contains pickup and dropoff timestamps.  
These variables can be used to create additional features such as trip duration, pickup hour, day of the week and month.

In [22]:
# Check datetime columns

datetime_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
]

for col in datetime_columns:
    if col in df_raw.columns:
        print(f"{col}: {df_raw[col].dtype}")
        print(f"Minimum: {df_raw[col].min()}")
        print(f"Maximum: {df_raw[col].max()}")
        print()
    else:
        print(f"{col} not found.")

tpep_pickup_datetime: datetime64[us]
Minimum: 2008-12-31 22:23:09
Maximum: 2022-05-18 20:41:57

tpep_dropoff_datetime: datetime64[us]
Minimum: 2008-12-31 23:06:56
Maximum: 2022-05-18 20:47:45



In [23]:
# Calculate raw trip duration in minutes for inspection

df_time_check = df_raw.copy()

df_time_check["trip_duration_minutes"] = (
    df_time_check["tpep_dropoff_datetime"] - df_time_check["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df_time_check["trip_duration_minutes"].describe()

count    2.463931e+06
mean     1.421220e+01
std      4.644531e+01
min     -3.442400e+03
25%      6.316667e+00
50%      1.018333e+01
75%      1.616667e+01
max      8.513183e+03
Name: trip_duration_minutes, dtype: float64

In [24]:
# Count invalid trip durations

invalid_duration = df_time_check[
    (df_time_check["trip_duration_minutes"].isna()) |
    (df_time_check["trip_duration_minutes"] <= 0)
]

print(f"Invalid trip duration rows: {len(invalid_duration):,}")
print(f"Percentage: {len(invalid_duration) / len(df_time_check) * 100:.2f}%")

Invalid trip duration rows: 2,449
Percentage: 0.10%


## 9. Data cleaning strategy

The following cleaning rules are applied:

1. Remove rows with missing target values.
2. Keep only positive `fare_amount` values.
3. Remove extremely high fares.
4. Keep only positive `trip_distance` values.
5. Remove extremely high trip distances.
6. Keep valid passenger counts.
7. Keep trips with reasonable duration.
8. Create useful time-based features.

These rules reduce noise and remove impossible or highly abnormal records.

In [25]:
def clean_taxi_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean NYC Yellow Taxi data using simple and explainable rules.
    """
    
    df = df.copy()
    
    # Normalize column names
    df.columns = [col.strip() for col in df.columns]
    
    # Remove missing target values
    df = df[df["fare_amount"].notna()]
    
    # Keep reasonable fare amounts
    df = df[df["fare_amount"] > 0]
    df = df[df["fare_amount"] < 300]
    
    # Keep reasonable trip distances
    df = df[df["trip_distance"].notna()]
    df = df[df["trip_distance"] > 0]
    df = df[df["trip_distance"] < 200]
    
    # Passenger count cleaning
    if "passenger_count" in df.columns:
        df = df[df["passenger_count"].notna()]
        df = df[df["passenger_count"] > 0]
        df = df[df["passenger_count"] <= 8]
    
    # Trip duration
    df["trip_duration_minutes"] = (
        df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    
    df = df[df["trip_duration_minutes"].notna()]
    df = df[df["trip_duration_minutes"] > 0]
    df = df[df["trip_duration_minutes"] <= 24 * 60]
    
    # Time-based features
    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
    df["pickup_dayofweek"] = df["tpep_pickup_datetime"].dt.dayofweek
    df["pickup_month"] = df["tpep_pickup_datetime"].dt.month
    
    return df

In [26]:
# Apply cleaning and measure execution time

start_time = time.perf_counter()

df_clean = clean_taxi_data(df_raw)

end_time = time.perf_counter()
cleaning_time = end_time - start_time

print(f"Cleaning completed.")
print(f"Cleaning time: {cleaning_time:.4f} seconds")
print(f"Original shape: {df_raw.shape}")
print(f"Cleaned shape: {df_clean.shape}")
print(f"Rows removed: {len(df_raw) - len(df_clean):,}")
print(f"Percentage removed: {(len(df_raw) - len(df_clean)) / len(df_raw) * 100:.2f}%")

Cleaning completed.
Cleaning time: 1.0162 seconds
Original shape: (2463931, 19)
Cleaned shape: (2301788, 23)
Rows removed: 162,143
Percentage removed: 6.58%


In [27]:
# Check cleaned dataset

df_clean.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,trip_duration_minutes,pickup_hour,pickup_dayofweek,pickup_month
0,1,2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.80,1.0,N,142,236,1,...,3.65,0.0,0.3,21.95,2.5,0.0,17.816667,0,5,1
1,1,2022-01-01 00:33:43,2022-01-01 00:42:07,1.0,2.10,1.0,N,236,42,1,...,4.00,0.0,0.3,13.30,0.0,0.0,8.400000,0,5,1
2,2,2022-01-01 00:53:21,2022-01-01 01:02:19,1.0,0.97,1.0,N,166,166,1,...,1.76,0.0,0.3,10.56,0.0,0.0,8.966667,0,5,1
3,2,2022-01-01 00:25:21,2022-01-01 00:35:23,1.0,1.09,1.0,N,114,68,2,...,0.00,0.0,0.3,11.80,2.5,0.0,10.033333,0,5,1
4,2,2022-01-01 00:36:48,2022-01-01 01:14:20,1.0,4.30,1.0,N,68,163,1,...,3.00,0.0,0.3,30.30,2.5,0.0,37.533333,0,5,1


In [28]:
# Descriptive statistics after cleaning

df_clean.describe().T

,count,mean,min,25%,50%,75%,max,std
VendorID,2301788.0,1.712722,1.0,1.0,2.0,2.0,2.0,0.452493
tpep_pickup_datetime,2301788,2022-01-17 00:06:42.012796,2008-12-31 22:23:09,2022-01-09 14:55:33,2022-01-17 10:45:36,2022-01-24 12:23:28.500000,2022-05-18 20:41:57,NaN
tpep_dropoff_datetime,2301788,2022-01-17 00:20:56.229757,2008-12-31 23:06:56,2022-01-09 15:09:18,2022-01-17 10:56:53,2022-01-24 12:36:36.250000,2022-05-18 20:47:45,NaN
passenger_count,2301788.0,1.422486,1.0,1.0,1.0,1.0,8.0,0.974578
trip_distance,2301788.0,3.142232,0.01,1.07,1.75,3.11,198.6,4.133938
RatecodeID,2301788.0,1.349142,1.0,1.0,1.0,1.0,99.0,5.45943
PULocationID,2301788.0,166.077108,1.0,132.0,162.0,234.0,265.0,65.056045
DOLocationID,2301788.0,163.772221,1.0,113.0,162.0,236.0,265.0,70.6626
payment_type,2301788.0,1.214518,1.0,1.0,1.0,1.0,5.0,0.42443
fare_amount,2301788.0,12.638934,0.01,6.5,9.0,13.5,299.0,11.523439


## 10. Feature selection for machine learning

For the machine learning pipeline, only columns that are useful, numerical or easy to encode are selected.

The target variable is:

`fare_amount`

The selected features include:

- passenger count
- trip distance
- pickup and dropoff location IDs
- payment type
- rate code
- trip duration
- pickup hour
- pickup day of week
- pickup month

In [29]:
# Select columns for the machine learning dataset

selected_columns = [
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "RatecodeID",
    "trip_duration_minutes",
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "fare_amount"
]

available_columns = [col for col in selected_columns if col in df_clean.columns]

df_ml = df_clean[available_columns].dropna()

print("Selected columns:")
print(available_columns)
print()
print(f"ML dataset shape: {df_ml.shape}")

Selected columns:
['passenger_count', 'trip_distance', 'PULocationID', 'DOLocationID', 'payment_type', 'RatecodeID', 'trip_duration_minutes', 'pickup_hour', 'pickup_dayofweek', 'pickup_month', 'fare_amount']

ML dataset shape: (2301788, 11)


In [30]:
# Check final ML dataset

df_ml.head()

,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,RatecodeID,trip_duration_minutes,pickup_hour,pickup_dayofweek,pickup_month,fare_amount
0,2.0,3.80,142,236,1,1.0,17.816667,0,5,1,14.5
1,1.0,2.10,236,42,1,1.0,8.400000,0,5,1,8.0
2,1.0,0.97,166,166,1,1.0,8.966667,0,5,1,7.5
3,1.0,1.09,114,68,2,1.0,10.033333,0,5,1,8.0
4,1.0,4.30,68,163,1,1.0,37.533333,0,5,1,23.5


In [31]:
# Confirm there are no missing values in the ML dataset

df_ml.isna().sum()

passenger_count          0
trip_distance            0
PULocationID             0
DOLocationID             0
payment_type             0
RatecodeID               0
trip_duration_minutes    0
pickup_hour              0
pickup_dayofweek         0
pickup_month             0
fare_amount              0
dtype: int64

## 11. Create dataset versions

To support the benchmark experiments, three dataset versions are created:

1. **Small dataset** — sample of 100,000 rows.
2. **Medium dataset** — full cleaned January 2022 dataset.
3. **ML dataset** — selected columns for model training.

The assignment also asks for experiments with smaller and larger datasets.  
The small dataset can be used as the smaller version, while later notebooks can add more months to create a larger version.

In [32]:
# Create a small sample for faster experiments

sample_size = min(100_000, len(df_clean))

df_small = df_clean.sample(
    n=sample_size,
    random_state=42
)

print(f"Small dataset shape: {df_small.shape}")

Small dataset shape: (100000, 23)


## 12. Save processed datasets

The processed datasets are saved in Parquet format because this format is efficient for analytics and Big Data libraries.

In [34]:
# Define output paths

clean_full_path = PROCESSED_DIR / "yellow_tripdata_2022-01_clean.parquet"
small_path = PROCESSED_DIR / "yellow_tripdata_2022-01_small_100k.parquet"
ml_path = PROCESSED_DIR / "yellow_tripdata_2022-01_ml.parquet"

In [35]:
# Save processed files and measure time

start_time = time.perf_counter()

df_clean.to_parquet(clean_full_path, index=False)
df_small.to_parquet(small_path, index=False)
df_ml.to_parquet(ml_path, index=False)

end_time = time.perf_counter()
save_time = end_time - start_time

print("Processed files saved successfully.")
print(f"Saving time: {save_time:.4f} seconds")
print()
print(clean_full_path)
print(small_path)
print(ml_path)

Processed files saved successfully.
Saving time: 0.7797 seconds

/Users/nunoantunes/Desktop/nyc_taxi_project/data/processed/yellow_tripdata_2022-01_clean.parquet
/Users/nunoantunes/Desktop/nyc_taxi_project/data/processed/yellow_tripdata_2022-01_small_100k.parquet
/Users/nunoantunes/Desktop/nyc_taxi_project/data/processed/yellow_tripdata_2022-01_ml.parquet


In [36]:
# Verify saved files

for path in [clean_full_path, small_path, ml_path]:
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{path.name}: {size_mb:.2f} MB")

yellow_tripdata_2022-01_clean.parquet: 48.42 MB
yellow_tripdata_2022-01_small_100k.parquet: 2.92 MB
yellow_tripdata_2022-01_ml.parquet: 15.79 MB


## 13. Save preprocessing summary

A small summary table is created to document the number of rows before and after cleaning.  
This table can later be used in the report.

In [38]:
# Create preprocessing summary

preprocessing_summary = pd.DataFrame({
    "stage": [
        "raw_dataset",
        "cleaned_dataset",
        "small_dataset",
        "ml_dataset"
    ],
    "rows": [
        len(df_raw),
        len(df_clean),
        len(df_small),
        len(df_ml)
    ],
    "columns": [
        df_raw.shape[1],
        df_clean.shape[1],
        df_small.shape[1],
        df_ml.shape[1]
    ]
})

preprocessing_summary["row_percentage_vs_raw"] = (
    preprocessing_summary["rows"] / len(df_raw) * 100
).round(2)

preprocessing_summary

,stage,rows,columns,row_percentage_vs_raw
0,raw_dataset,2463931,19,100.00
1,cleaned_dataset,2301788,23,93.42
2,small_dataset,100000,23,4.06
3,ml_dataset,2301788,11,93.42


In [39]:
# Save summary to CSV

summary_path = RESULTS_DIR / "preprocessing_summary.csv"

preprocessing_summary.to_csv(summary_path, index=False)

print(f"Preprocessing summary saved to: {summary_path}")

Preprocessing summary saved to: /Users/nunoantunes/Desktop/nyc_taxi_project/results/preprocessing_summary.csv


## 14. Final notes

The raw NYC Yellow Taxi January 2022 dataset was successfully loaded, inspected, cleaned and saved.

The main outputs of this notebook are:

- `yellow_tripdata_2022-01_clean.parquet`
- `yellow_tripdata_2022-01_small_100k.parquet`
- `yellow_tripdata_2022-01_ml.parquet`
- `preprocessing_summary.csv`

These files will be used in the next notebooks for benchmarking and machine learning experiments.

In [40]:
print("Notebook 01 completed successfully.")
print("Ready for Notebook 02: Benchmark operations.")

Notebook 01 completed successfully.
Ready for Notebook 02: Benchmark operations.
